In [ ]:
import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # suppress TF info/warning logs
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import ConfusionMatrixDisplay

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow version : {tf.__version__}')
print('✅ All libraries imported!')

In [ ]:
CLASS_NAMES = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal',  'Shirt',   'Sneaker',  'Bag',   'Ankle boot']

print('⏳ Loading Fashion-MNIST ...')
(X_train_raw, y_train), (X_test_raw, y_test) = keras.datasets.fashion_mnist.load_data()

print('✅ Dataset loaded!')
print(f'   Train : {X_train_raw.shape}  →  {len(X_train_raw):,} images')
print(f'   Test  : {X_test_raw.shape}   →  {len(X_test_raw):,} images')
print(f'   Pixel range (raw) : [{X_train_raw.min()}, {X_train_raw.max()}]')

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
fig.suptitle('Fashion-MNIST — Sample Images (one per class)', fontsize=13, fontweight='bold')

for class_idx in range(10):
    # pick first image of each class from training set
    sample_idx = np.where(y_train == class_idx)[0][0]
    axes[0, class_idx].imshow(X_train_raw[sample_idx], cmap='gray')
    axes[0, class_idx].axis('off')
    axes[0, class_idx].set_title(CLASS_NAMES[class_idx], fontsize=8)

    # pick a second sample
    sample_idx2 = np.where(y_train == class_idx)[0][1]
    axes[1, class_idx].imshow(X_train_raw[sample_idx2], cmap='gray')
    axes[1, class_idx].axis('off')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved → sample_images.png')

In [ ]:
# Normalize to [0.0, 1.0]
X_train = X_train_raw.astype('float32') / 255.0
X_test  = X_test_raw.astype('float32')  / 255.0

# Reshape: (N, 28, 28) → (N, 28, 28, 1)  ← channel dimension
X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

print('✅ Preprocessing complete!')
print(f'   CNN input shape : {X_train.shape}   ← 28×28 grid, 1 channel')
print(f'   Pixel range     : [{X_train.min():.1f}, {X_train.max():.1f}]')

In [ ]:
cnn_model = keras.Sequential([

    layers.Input(shape=(28, 28, 1)),

    # ── Block 1: detect simple patterns (edges, lines) ──────────
    layers.Conv2D(32, (3, 3), padding='same'),
    layers.BatchNormalization(),      # normalize → stable training
    layers.Activation('relu'),
    layers.MaxPooling2D(2, 2),        # 28×28 → 14×14
    layers.Dropout(0.2),

    # ── Block 2: detect medium patterns (shapes, curves) ────────
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2, 2),        # 14×14 → 7×7
    layers.Dropout(0.25),

    # ── Block 3: detect complex patterns (clothing features) ─────
    layers.Conv2D(128, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.3),              # no pooling — preserve 7×7 detail

    # ── Classifier head ──────────────────────────────────────────
    layers.Flatten(),                 # 7×7×128 = 6,272 values
    layers.Dense(128, activation='relu',
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.5),
    layers.Dense(64, activation='relu',
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax'),  # 10 classes

], name='CNN_BatchNorm')

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # integer labels
    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

print('🚀 Training CNN ...')
print('   Conv layers preserve 2D spatial structure → expect strong accuracy!\n')

history = cnn_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=1
)

print('\n✅ CNN training complete!')

In [ ]:
# Method 1: .evaluate()
test_loss, test_acc = cnn_model.evaluate(X_test, y_test, verbose=0)

print('=' * 42)
print('   CNN — Test Set Results')
print('=' * 42)
print(f'   Test Loss     : {test_loss:.4f}')
print(f'   Test Accuracy : {test_acc * 100:.2f}%')
print('=' * 42)

# Method 2: .predict()
probs = cnn_model.predict(X_test, verbose=0)   # shape: (10000, 10)
preds = probs.argmax(axis=1)                   # predicted class per image

print(f'\n.predict() output shape : {probs.shape}')
print(f'First 10 predicted       : {preds[:10]}')
print(f'First 10 actual          : {y_test[:10]}')
print(f'Manual accuracy          : {(preds == y_test).mean() * 100:.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('CNN Training Progress', fontsize=14, fontweight='bold')

# Loss
axes[0].plot(history.history['loss'],     label='Train',      color='steelblue', lw=2)
axes[0].plot(history.history['val_loss'], label='Validation', color='tomato',    lw=2, linestyle='--')
axes[0].set_title('Loss (Cross-Entropy) over Epochs', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(history.history['accuracy'],     label='Train',      color='steelblue', lw=2)
axes[1].plot(history.history['val_accuracy'], label='Validation', color='tomato',    lw=2, linestyle='--')
axes[1].axhline(test_acc, color='green', linestyle=':', lw=1.5, label=f'Test Acc ({test_acc*100:.1f}%)')
axes[1].set_title('Accuracy over Epochs', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → training_curves.png')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

ConfusionMatrixDisplay.from_predictions(
    y_test, preds,
    display_labels=CLASS_NAMES,
    ax=ax,
    colorbar=False,
    xticks_rotation=45
)

ax.set_title(
    f'CNN Confusion Matrix  (Test Accuracy: {test_acc*100:.1f}%)\n'
    'Diagonal = correct  |  Off-diagonal = mistakes',
    fontsize=12, fontweight='bold'
)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → confusion_matrix.png')

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
fig.suptitle('Sample Predictions  |  Green = correct   Red = wrong',
             fontsize=12, fontweight='bold')

for i in range(10):
    img = X_test[i].reshape(28, 28)
    color = 'green' if preds[i] == y_test[i] else 'red'
    conf  = probs[i, preds[i]] * 100

    # Top row: image
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].axis('off')
    axes[0, i].set_title(CLASS_NAMES[preds[i]], fontsize=8, color=color)

    # Bottom row: confidence bar
    axes[1, i].bar(range(10), probs[i], color='steelblue', alpha=0.7)
    axes[1, i].bar(preds[i], probs[i, preds[i]], color=color)
    axes[1, i].set_xticks([])
    axes[1, i].set_yticks([])
    axes[1, i].set_title(f'{conf:.0f}%', fontsize=8)

axes[0, 0].set_ylabel('Image',      fontsize=10, labelpad=6)
axes[1, 0].set_ylabel('Confidence', fontsize=10, labelpad=6)

plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → sample_predictions.png')

In [ ]:
MODEL_PATH = 'fashion_cnn_model.keras'
cnn_model.save(MODEL_PATH)
print(f'✅ Model saved → {MODEL_PATH}')

# Verify reload
loaded = keras.models.load_model(MODEL_PATH)
reload_loss, reload_acc = loaded.evaluate(X_test, y_test, verbose=0)
print(f'✅ Reload verified — accuracy: {reload_acc * 100:.2f}%  (matches original)')